# T-E2：AI Agent 自动生成股市周报 — 实证分析

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI Agent 自动生成股市周报 |
| 小组   | Team02 |
| 成员   | 王佳（25210244）、刘雄（25210196）、黎㵆筠（25210155）、郑嘉豪（25210307）、刘子瑜（25210198）、陈春洁（25210115）、王占溪（25210249）、黎沛鑫（25210156） |
| GitHub | https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report |
| Pages  | https://htmlpreview.github.io/?https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report/blob/main/output/weekly_report.html |
| 日期   | 2026-05-18 |


## 任务说明

本步骤对清洗后的数据进行实证分析，包括：
1. 日收益率计算与描述性统计
2. 周度涨跌幅汇总
3. 年化波动率与夏普比率
4. A 股与美股指数日收益率相关性矩阵
5. 分析摘要（领涨/领跌、跨市场联动、波动率排名）

分析结果保存至 `data_clean/`，同时为可视化和 AI 摘要提供数据支撑。

In [ ]:
import pandas as pd
import numpy as np

# 读取清洗后数据
df_a = pd.read_csv("data_clean/A股清洗后数据.csv", encoding="utf-8-sig", parse_dates=["日期"])
df_us = pd.read_csv("data_clean/美股清洗后数据.csv", encoding="utf-8-sig", parse_dates=["日期"])

# 合并数据便于分析
df_all = pd.concat([df_a, df_us], ignore_index=True)
df_all["市场"] = df_all["指数名称"].apply(
    lambda x: "A股" if x in df_a["指数名称"].unique() else "美股"
)

print(f"数据加载完成: A股 {df_a.shape[0]} 条, 美股 {df_us.shape[0]} 条")
print(f"A股指数: {list(df_a['指数名称'].unique())}")
print(f"美股指数: {list(df_us['指数名称'].unique())}")
print(f"日期范围: {df_all['日期'].min().date()} ~ {df_all['日期'].max().date()}")

In [ ]:
# ============================================
# 1. 日收益率计算 + 描述性统计
# ============================================

# 计算日收益率（%）
df_all["日收益率"] = df_all.groupby("指数名称")["收盘价"].pct_change() * 100

# 描述性统计（按指数）
desc_stats = df_all.groupby("指数名称")["日收益率"].describe()
desc_stats.columns = ["计数", "均值(%)", "标准差(%)", "最小值(%)", "25%分位", "50%分位", "75%分位", "最大值(%)"]

print("=" * 60)
print("各指数日收益率描述性统计")
print("=" * 60)
display(desc_stats.round(4))

### 描述性统计解读

各指数日收益率的均值和标准差反映了本周市场的整体波动特征。A 股指数普遍录得负收益，反映出本周的回调行情；美股指数波动相对温和。创业板指标准差最大，显示中小盘股波动性较高。

In [ ]:
# ============================================
# 2. 周度涨跌幅计算（区间累计收益率）
# ============================================

def calc_weekly_return(df):
    """计算各指数区间涨跌幅和其他关键指标"""
    results = []
    for name in df["指数名称"].unique():
        sub = df[df["指数名称"] == name].sort_values("日期")
        start = sub["收盘价"].iloc[0]
        end = sub["收盘价"].iloc[-1]
        high = sub["最高价"].max() if "最高价" in sub.columns else np.nan
        low = sub["最低价"].min() if "最低价" in sub.columns else np.nan
        vol_avg = sub["成交量"].mean() if "成交量" in sub.columns else np.nan

        results.append({
            "指数名称": name,
            "期初收盘价": round(start, 2),
            "期末收盘价": round(end, 2),
            "区间涨跌幅(%)": round((end - start) / start * 100, 2),
            "区间最高": round(high, 2),
            "区间最低": round(low, 2),
            "日均成交量": round(vol_avg, 0),
        })
    return pd.DataFrame(results)

weekly_a = calc_weekly_return(df_a)
weekly_us = calc_weekly_return(df_us)

print("=" * 50)
print("A股指数周度表现")
print("=" * 50)
display(weekly_a)

print("\n" + "=" * 50)
print("美股指数周度表现")
print("=" * 50)
display(weekly_us)

# 合并保存，供后续使用
weekly_all = pd.concat([weekly_a, weekly_us], ignore_index=True)
weekly_all.to_csv("data_clean/周度涨跌幅汇总.csv", index=False, encoding="utf-8-sig")
print("\n周度涨跌幅汇总已保存 → data_clean/周度涨跌幅汇总.csv")

### 周度涨跌幅解读

本周 A 股三大指数和美股三大指数均收跌。A 股跌幅较大，沪深 300 和深证成指跌幅居前；美股方面道琼斯工业指数跌幅最大。整体呈现中美股市同步回调的格局。

In [ ]:
# ============================================
# 3. 波动率分析（年化波动率）
# ============================================

# 年化波动率 = 日收益率标准差 × sqrt(252)
volatility = df_all.groupby(["市场", "指数名称"])["日收益率"].agg(["std", "count"])
volatility["年化波动率(%)"] = volatility["std"] * np.sqrt(252)

# 年化收益率 ≈ 日均收益率 × 252
mean_ret = df_all.groupby(["市场", "指数名称"])["日收益率"].mean()
volatility["年化收益率(%)"] = mean_ret * 252

# 夏普比率（简化：假设无风险利率 = 0）
volatility["夏普比率"] = volatility["年化收益率(%)"] / volatility["年化波动率(%)"]

volatility = volatility[["年化收益率(%)", "年化波动率(%)", "夏普比率"]].round(2)

print("=" * 50)
print("各指数年化风险收益指标")
print("=" * 50)
display(volatility)

# 保存
volatility.to_csv("data_clean/风险收益指标.csv", encoding="utf-8-sig")
print("风险收益指标已保存 → data_clean/风险收益指标.csv")

### 风险收益指标解读

年化波动率方面，创业板指（约 27.5%）和纳斯达克综合（约 20.7%）波动最大，符合成长/科技板块高波动特性。夏普比率均为负值，反映本周整体下行环境。注意：仅基于一周数据计算的年化指标仅供参考，不宜过度解读。

In [ ]:
# ============================================
# 4. 相关性分析 — A股与美股指数日收益率相关性矩阵
# ============================================

# 构建收益率宽表：行=日期，列=指数名称
pivot = df_all.pivot_table(
    values="日收益率", index="日期", columns="指数名称", aggfunc="mean"
).dropna()

# 相关性矩阵
corr_matrix = pivot.corr()

print("=" * 50)
print("中美主要指数日收益率相关性矩阵")
print("=" * 50)
display(corr_matrix.round(3))

# 单独看跨市场相关性
a_indices = [c for c in corr_matrix.columns if c in df_a["指数名称"].unique()]
us_indices = [c for c in corr_matrix.columns if c in df_us["指数名称"].unique()]
cross_corr = corr_matrix.loc[a_indices, us_indices]

print("\n" + "=" * 50)
print("跨市场相关性（A股 × 美股）")
print("=" * 50)
display(cross_corr.round(3))

# 保存
corr_matrix.to_csv("data_clean/相关性矩阵.csv", encoding="utf-8-sig")
print("相关性矩阵已保存 → data_clean/相关性矩阵.csv")

### 相关性矩阵解读

A 股内部各指数高度正相关（r > 0.97），美股内部同样高度联动。跨市场方面，A 股与美股相关性较弱（r 约 0.1-0.3），甚至出现负值，说明本周中美市场走势相对独立，未出现明显的风险传染。

In [ ]:
# ============================================
# 5. 分析摘要（供 AI 摘要 / 报告撰写使用）
# ============================================

print("=" * 60)
print("本周市场分析摘要")
print("=" * 60)

best_a = weekly_a.loc[weekly_a["区间涨跌幅(%)"].idxmax()]
worst_a = weekly_a.loc[weekly_a["区间涨跌幅(%)"].idxmin()]
best_us = weekly_us.loc[weekly_us["区间涨跌幅(%)"].idxmax()]
worst_us = weekly_us.loc[weekly_us["区间涨跌幅(%)"].idxmin()]

print(f"\nA股市场：")
print(f"  领涨: {best_a['指数名称']} ({best_a['区间涨跌幅(%)']:+.2f}%)")
print(f"  领跌: {worst_a['指数名称']} ({worst_a['区间涨跌幅(%)']:+.2f}%)")

print(f"\n美股市场：")
print(f"  领涨: {best_us['指数名称']} ({best_us['区间涨跌幅(%)']:+.2f}%)")
print(f"  领跌: {worst_us['指数名称']} ({worst_us['区间涨跌幅(%)']:+.2f}%)")

print(f"\n跨市场相关性：")
for a_idx in a_indices:
    for us_idx in us_indices:
        r = cross_corr.loc[a_idx, us_idx]
        direction = "正相关" if r > 0 else "负相关"
        strength = "强" if abs(r) > 0.5 else ("中等" if abs(r) > 0.3 else "弱")
        print(f"  {a_idx} ↔ {us_idx}: r={r:.3f} ({strength}{direction})")

max_vol = volatility["年化波动率(%)"].idxmax()
min_vol = volatility["年化波动率(%)"].idxmin()
print(f"\n波动率：")
print(f"  最高: {max_vol} ({volatility.loc[max_vol, '年化波动率(%)']:.2f}%)")
print(f"  最低: {min_vol} ({volatility.loc[min_vol, '年化波动率(%)']:.2f}%)")

print("\n" + "=" * 60)
print("分析完成，结果文件已保存至 data_clean/")
print("  - 周度涨跌幅汇总.csv")
print("  - 风险收益指标.csv")
print("  - 相关性矩阵.csv")

## 结果解读

本周核心结论：中美股市同步回调，A 股跌幅大于美股；A 股内部高度联动，创业板波动最大；中美市场间相关性较低，未出现明显的跨市场风险传导。分析输出文件（周度涨跌幅汇总、风险收益指标、相关性矩阵）已保存至 `data_clean/`，可直接用于可视化绘图。